# Minimal Training Test (~1 min)

Just verifies training works and losses decrease.

In [ ]:
import torch
import lightning as L
from pathlib import Path
from ase.build import bulk
from ase.io import write

from NPS.logp.models import LitLogPModel
from NPS.logp.data import PeriodicStructureDataModule

In [ ]:
# Create tiny dataset
data_dir = Path("tiny_data")
data_dir.mkdir(exist_ok=True)

write(data_dir / 'bcc.extxyz', bulk('Fe', 'bcc', a=2.87, cubic=True) * (3,3,3))
write(data_dir / 'fcc.extxyz', bulk('Cu', 'fcc', a=3.61, cubic=True) * (3,3,3))

structure_types = ["bcc", "fcc"]

In [ ]:
# DataModule
dm = PeriodicStructureDataModule(
    file_list=[str(data_dir / f"{s}.extxyz") for s in structure_types],
    cutoff=4.0,
    duplicate=20,
    batch_size=2,
    num_workers=0,
    structure_types=structure_types,
)

# Tiny model
model = LitLogPModel(
    nclass=2,
    cutoff=4.0,
    hidden_irreps="16x0e+16x1o+16x2e",
    num_interactions=1,
    sigma_max=0.15,
    sigma_logp_scale=0.15,
    learn_rate=1e-3,
)

print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Train
trainer = L.Trainer(accelerator='cpu', max_epochs=3, enable_checkpointing=False, logger=False)
trainer.fit(model, dm)
print("Done!")